# ⚙️ Notebook 2: Feature Engineering

## Learning Objectives
By the end of this notebook, you will:
- Understand what features are and why they matter
- Create technical indicators from price data
- Build a sentiment proxy
- Create the target variable for prediction
- Prepare data for machine learning

## What You'll Learn
- Rolling window calculations
- Technical analysis indicators
- Avoiding look-ahead bias
- Feature importance concepts

---

## What are Features?

**Features** are the input variables that a machine learning model uses to make predictions.

**Raw data** (Close price: $150) → **Features** (5-day average: $148, momentum: +$5) → **Model** → **Prediction** (Up/Down)

### Why Not Just Use Raw Prices?
- Absolute prices don't tell us about trends
- We need context: Is the price rising or falling?
- Features capture patterns that predict future movement

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.append('stock_ml_project/..')
import config
from src.data_collection import load_stock_data
from src.preprocessing import preprocess_stock_data
from src import feature_engineering as fe

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Ready to engineer features!")

## Step 1: Load and Preprocess Data

In [ ]:
# Load the data from Notebook 1
ticker = 'AAPL'
raw_data = load_stock_data(ticker, 'raw')

if raw_data is None:
    print("❌ Data not found. Please run Notebook 1 first!")
else:
    # Preprocess
    data = preprocess_stock_data(raw_data)
    print(f"✅ Loaded {len(data)} days of {ticker} data")
    print(f"Date range: {data['Date'].min()} to {data['Date'].max()}")
    
    # Show first few rows
    data.head()

## Step 2: Feature 1 - Daily Returns

Returns show percentage change in price.

**Formula**: `(Today's Close - Yesterday's Close) / Yesterday's Close`

In [ ]:
# Calculate returns
data = fe.calculate_returns(data)

# Visualize
print("Returns Statistics:")
print(f"Mean: {data['Returns'].mean():.4f}")
print(f"Std: {data['Returns'].std():.4f}")

# Show sample
data[['Date', 'Close', 'Returns']].head(10)

## Step 3: Feature 2 - Moving Averages

Moving averages smooth out price fluctuations to show trends.

- **SMA_5**: Average of last 5 days (short-term trend)
- **SMA_20**: Average of last 20 days (long-term trend)

**Trading signal**: When short-term > long-term, it's a bullish signal!

In [ ]:
# Add moving averages
data = fe.add_rolling_averages(data, windows=[5, 10, 20])

# Visualize
plt.figure(figsize=(14, 6))
plt.plot(data['Date'], data['Close'], label='Close Price', linewidth=2, alpha=0.7)
plt.plot(data['Date'], data['SMA_5'], label='5-day SMA', linewidth=1.5)
plt.plot(data['Date'], data['SMA_10'], label='10-day SMA', linewidth=1.5)
plt.plot(data['Date'], data['SMA_20'], label='20-day SMA', linewidth=1.5)
plt.title('Price vs Moving Averages', fontsize=16)
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Notice how moving averages smooth out daily noise!")
print("When SMA_5 crosses above SMA_20, it often signals an uptrend.")

## Step 4: Feature 3 - Volatility

Volatility measures how much the price fluctuates.

**Formula**: Standard deviation of returns over a rolling window

In [ ]:
# Calculate volatility
data = fe.calculate_volatility(data, window=20)

# Visualize
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Price
ax1.plot(data['Date'], data['Close'], linewidth=2)
ax1.set_title('Price and Volatility', fontsize=16)
ax1.set_ylabel('Price ($)')
ax1.grid(True, alpha=0.3)

# Volatility
ax2.plot(data['Date'], data['Volatility'], color='red', linewidth=2)
ax2.set_xlabel('Date')
ax2.set_ylabel('Volatility')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 High volatility = more uncertainty = harder to predict!")

## Step 5: More Features

Let's add several more features at once:

In [ ]:
# Add all remaining features
data = fe.calculate_momentum(data)
data = fe.calculate_volume_change(data)
data = fe.calculate_hl_spread(data)

print("Added features:")
print("- Momentum: Price change over N days")
print("- Volume_Change: Change in trading volume")
print("- HL_Spread: High-Low range normalized")

# Show current columns
print(f"\nTotal columns now: {len(data.columns)}")
print(f"Columns: {list(data.columns)}")

## Step 6: Sentiment Proxy

We create a price-based sentiment indicator as a proxy for market sentiment.

This is a simplified version. In reality, you'd use NLP on news articles!

In [ ]:
# Create sentiment features
data = fe.create_sentiment_proxy(data)
data = fe.smooth_sentiment(data)

# Visualize sentiment
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Price
ax1.plot(data['Date'], data['Close'], linewidth=2)
ax1.set_title('Price and Sentiment', fontsize=16)
ax1.set_ylabel('Price ($)')
ax1.grid(True, alpha=0.3)

# Sentiment
ax2.plot(data['Date'], data['Sentiment_Proxy'], alpha=0.5, label='Raw Sentiment')
ax2.plot(data['Date'], data['Sentiment_SMA'], linewidth=2, label='Smoothed Sentiment')
ax2.axhline(y=0, color='black', linestyle='--', alpha=0.3)
ax2.set_xlabel('Date')
ax2.set_ylabel('Sentiment (-1 to 1)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Positive sentiment (>0) often correlates with price increases!")

## Step 7: THE TARGET VARIABLE ⚠️

This is what we're trying to predict!

**Target = 1** if tomorrow's price > today's price (UP)
**Target = 0** if tomorrow's price ≤ today's price (DOWN)

### CRITICAL: Avoiding Look-Ahead Bias
We use `.shift(-1)` to get tomorrow's price. This creates NaN in the last row because there's no "tomorrow" data yet!

In [ ]:
# Create target variable
data = fe.create_target_variable(data, horizon=1)

# Show examples
print("Target Variable Examples:")
print("="*70)
sample = data[['Date', 'Close', 'Future_Close', 'Target']].tail(10)
print(sample)

# Distribution
target_counts = data['Target'].value_counts()
print(f"\nTarget Distribution:")
print(f"Down (0): {target_counts.get(0, 0)} days")
print(f"Up (1): {target_counts.get(1, 0)} days")

# Visualize
plt.figure(figsize=(8, 6))
data['Target'].value_counts().plot(kind='bar', color=['red', 'green'])
plt.title('Target Variable Distribution', fontsize=16)
plt.xlabel('Direction')
plt.ylabel('Count')
plt.xticks([0, 1], ['Down (0)', 'Up (1)'], rotation=0)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Step 8: Feature Correlation Analysis

Let's see which features correlate with our target variable!

In [ ]:
# Calculate correlations with target
feature_cols = ['Returns', 'SMA_5', 'SMA_10', 'SMA_20', 'Volatility', 
                'Momentum', 'Volume_Change', 'HL_Spread', 
                'Sentiment_Proxy', 'Sentiment_SMA']

correlations = data[feature_cols + ['Target']].corr()['Target'].drop('Target').sort_values()

# Visualize
plt.figure(figsize=(10, 6))
correlations.plot(kind='barh', color=['red' if x < 0 else 'green' for x in correlations])
plt.title('Feature Correlation with Target', fontsize=16)
plt.xlabel('Correlation Coefficient')
plt.axvline(x=0, color='black', linestyle='--', alpha=0.5)
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\n💡 Features with higher correlation are more predictive!")
print(f"\nStrongest positive correlation: {correlations.idxmax()} ({correlations.max():.4f})")
print(f"Strongest negative correlation: {correlations.idxmin()} ({correlations.min():.4f})")

## Step 9: Prepare Data for Machine Learning

In [ ]:
# Prepare ML data
X, y, feature_names = fe.prepare_ml_data(data)

print("\n🎯 Data Ready for Machine Learning!")
print("="*70)
print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")
print(f"\nFeature names: {feature_names}")

# Show sample
print("\nSample of prepared data:")
sample_df = pd.DataFrame(X, columns=feature_names).head()
sample_df['Target'] = y.head().values
sample_df

## Step 10: Save Engineered Features

In [ ]:
# Save to CSV
output_path = config.get_data_path(ticker, 'features')
data.to_csv(output_path, index=False)
print(f"✅ Saved engineered features to: {output_path}")

print("\n📊 Summary:")
print(f"Original columns: 7 (Date, Open, High, Low, Close, Volume, Ticker)")
print(f"Final columns: {len(data.columns)}")
print(f"New features created: {len(data.columns) - 7}")

## 🎯 Exercise: Create Your Own Feature!

Try implementing the **Relative Strength Index (RSI)**:

RSI measures the speed and magnitude of price changes.
- RSI > 70: Overbought (might go down)
- RSI < 30: Oversold (might go up)

In [ ]:
# YOUR CODE HERE
# Implement RSI calculation

def calculate_rsi(data, window=14):
    """
    Calculate Relative Strength Index
    
    Hint:
    1. Calculate price changes (delta)
    2. Separate gains and losses
    3. Calculate average gain and average loss
    4. RS = average gain / average loss
    5. RSI = 100 - (100 / (1 + RS))
    """
    # Your implementation here
    pass

# Test it
# data['RSI'] = calculate_rsi(data)
# Plot it alongside price

## 📝 Key Takeaways

1. **Features** transform raw data into predictive signals
2. **Rolling windows** capture trends over time
3. **Technical indicators** (SMA, volatility) are proven predictive features
4. **Target variable** is binary: up (1) or down (0)
5. **Look-ahead bias** is a common mistake - never use future data!
6. **Feature correlation** helps identify the most predictive variables

## Next Steps

In Notebook 3, we'll train a Random Forest model using these features!

---

**Questions to think about:**
- Why do moving averages work as predictive features?
- What other features could we create from price and volume?
- How might we incorporate news sentiment in the future?